<a href="https://colab.research.google.com/github/rajidush/RP-J26-DS-316/blob/c3%2Fmodel-tuning/component3_socratic_educator/notebooks/draft_inquire_dataset_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Draft INQUIRE fine-tuning examples (Component 3)

Runs `draft_inquire_dataset.py` on Colab with Gemini through Colab's built-in models and distilabel.
Outputs are **drafts**: every row is reviewed by the owner before it becomes a seed example.

Order matters: **scenarios** (freezes the 10 test scenarios) → **replies** → **drafts**.
Each stage is resumable: if one fails part-way, re-run the same cell.
Outputs go to Google Drive so a runtime disconnect loses nothing.

In [1]:
!pip -q install "distilabel[openai]==1.5.3"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.2/511.2 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.5/83.5 kB 6.3 MB/s eta 0:00:00


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Upload `draft_inquire_dataset.py` and `data/educator_seed.jsonl` from `component3_socratic_educator/`.

In [21]:
import os
from google.colab import files

uploaded = files.upload()

# Look for the uploaded files, allowing for automatically renamed duplicates (like 'file (1).py')
draft_file = next((f for f in uploaded if f.startswith("draft_inquire_dataset") and f.endswith(".py")), None)
seed_file = next((f for f in uploaded if f.startswith("educator_seed") and f.endswith(".jsonl")), None)

assert draft_file and seed_file, f"Expected files not found in uploaded list: {list(uploaded.keys())}"

# If renamed, copy or rename them to the expected standard filenames
if draft_file != "draft_inquire_dataset.py":
    os.replace(draft_file, "draft_inquire_dataset.py")
if seed_file != "educator_seed.jsonl":
    os.replace(seed_file, "educator_seed.jsonl")

print("Files successfully uploaded and prepared: draft_inquire_dataset.py, educator_seed.jsonl")

KeyboardInterrupt: 

In [6]:
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/c3_inquire_drafts"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Check model access. This also loads the Colab model-proxy credentials into the environment for the script.

In [23]:
import os
from google.colab import ai
MODEL = "google/gemini-3.1-pro-preview"   # fallback: "google/gemini-2.5-pro"
print(ai.generate_text("Reply with the single word OK.", model_name=MODEL))
assert os.environ.get("MODEL_PROXY_HOST"), "MODEL_PROXY_HOST not set"
assert os.environ.get("MODEL_PROXY_API_KEY"), "MODEL_PROXY_API_KEY not set"

PermissionDeniedError: Error code: 403 - {'code': '', 'message': 'The max estimated cost of operation ($0.786448) exceeds your available quota (based on max_output_tokens). Set max_output_tokens in your request to lower the reservation, or try again later.', 'param': '', 'type': 'permission_error'}

## Stage 1: scenarios + frozen test set

In [10]:
!python draft_inquire_dataset.py scenarios --seed-file educator_seed.jsonl --out "{OUT}" --model "{MODEL}"

/usr/local/lib/python3.13/dist-packages/pydantic/_internal/_generate_schema.py:2274: UnsupportedFieldAttributeWarning: The 'default' attribute with value None was provided to the `Field()` function, which has no effect in the context it was used. 'default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/pydantic/_internal/_generate_schema.py:2274: UnsupportedFieldAttributeWarning: The 'default' attribute with value None was provided to the `Field()` function, which has no effect in the context it was used. 'default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the 

In [7]:
import json
frozen = json.load(open(f"{OUT}/scenarios.json", encoding="utf-8"))
for s in frozen["scenarios"]:
    mark = "TEST" if s["scenario_id"] in frozen["test_scenario_ids"] else "    "
    print(mark, s["scenario_id"], s["risk_category"].ljust(18), s["context"])
print("near-duplicates to review:", frozen["near_duplicates_for_review"])

     S027 explicit_visual    A risk trigger was received for an unsolicited explicit image sent to the child via a direct messaging app; the content was intercepted and hidden.
TEST S028 explicit_visual    A risk trigger was received for explicit user-generated artwork displayed on a custom avatar in a multiplayer sandbox game.
TEST S029 explicit_visual    A risk trigger was received for adult visual material masquerading as a children's animation on a video-sharing platform; playback was immediately blocked.
     S030 explicit_visual    A risk trigger was received for a sexually explicit banner advertisement loading on the sidebar of a free gaming website.
TEST S031 violence           A risk trigger was received for search queries regarding the construction of homemade weapons; the browser blocked the results.
     S032 violence           A risk trigger was received for detailed descriptions of physical violence typed into a collaborative school document.
TEST S033 violence           

## Stage 2: previous educator message + 5 child replies per scenario

In [12]:
!python draft_inquire_dataset.py replies --seed-file educator_seed.jsonl --out "{OUT}" --model "{MODEL}"

/usr/local/lib/python3.13/dist-packages/pydantic/_internal/_generate_schema.py:2274: UnsupportedFieldAttributeWarning: The 'default' attribute with value None was provided to the `Field()` function, which has no effect in the context it was used. 'default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/pydantic/_internal/_generate_schema.py:2274: UnsupportedFieldAttributeWarning: The 'default' attribute with value None was provided to the `Field()` function, which has no effect in the context it was used. 'default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the 

In [3]:
import os
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

from openai import OpenAI
c = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key=os.environ["GEMINI_API_KEY"])
for m in ["gemini-3.1-pro-preview", "gemini-2.5-pro", "gemini-2.5-flash"]:
    try:
        c.chat.completions.create(model=m, messages=[{"role": "user", "content": "Reply OK"}], max_tokens=1024)
        print(m, "WORKS")
    except Exception as e:
        print(m, "FAILS ->", str(e)[:120])


gemini-3.1-pro-preview FAILS -> Error code: 429 - [{'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billi
gemini-2.5-pro FAILS -> Error code: 404 - [{'error': {'code': 404, 'message': 'This model models/gemini-2.5-pro is no longer available to new us
gemini-2.5-flash FAILS -> Error code: 404 - [{'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new 


In [4]:
import os, draft_inquire_dataset as d
from openai import AsyncOpenAI
client = AsyncOpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
                     api_key=os.environ["GEMINI_API_KEY"])
create = d._without_unsupported_fields(client.chat.completions.create)
r = await create(model="gemini-3.8-flash", messages=[{"role": "user", "content": "Reply OK"}],
                 logprobs=False, top_logprobs=None, max_tokens=8192, n=1, frequency_penalty=0.0,
                 presence_penalty=0.0, temperature=0.9, top_p=1.0, stop=None, extra_body=None)
print(r.choices[0].message.content)


/usr/local/lib/python3.13/dist-packages/pydantic/_internal/_generate_schema.py:2274: UnsupportedFieldAttributeWarning: The 'default' attribute with value None was provided to the `Field()` function, which has no effect in the context it was used. 'default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/pydantic/_internal/_generate_schema.py:2274: UnsupportedFieldAttributeWarning: The 'default' attribute with value None was provided to the `Field()` function, which has no effect in the context it was used. 'default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the 

OK


In [31]:
models = sorted(m.id.removeprefix("models/") for m in c.models.list())
gen = [m for m in models if m.startswith("gemini") and not any(x in m for x in ("embed", "tts", "image", "audio", "live"))]
print(gen)
for m in gen:
    try:
        c.chat.completions.create(model=m, messages=[{"role": "user", "content": "Reply OK"}], max_tokens=1024)
        print(m, "WORKS")
    except Exception as e:
        print(m, "FAILS ->", str(e)[:90])


['gemini-2.5-computer-use-preview-10-2025', 'gemini-2.5-flash', 'gemini-2.5-flash-lite', 'gemini-2.5-pro', 'gemini-3-flash-preview', 'gemini-3.1-flash-lite', 'gemini-3.1-flash-lite-preview', 'gemini-3.1-pro-preview', 'gemini-3.1-pro-preview-customtools', 'gemini-3.5-flash', 'gemini-3.5-flash-lite', 'gemini-3.5-transcribe', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'gemini-flash-latest', 'gemini-flash-lite-latest', 'gemini-nano-banana-2.1', 'gemini-omni-1.1-flash', 'gemini-omni-flash-preview', 'gemini-pro-latest', 'gemini-robotics-er-2-preview', 'gemini-robotics-er-2-streaming-preview']
gemini-2.5-computer-use-preview-10-2025 FAILS -> Error code: 429 - [{'error': {'code': 429, 'message': 'You exceeded your current quota, pl
gemini-2.5-flash FAILS -> Error code: 404 - [{'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash 
gemini-2.5-flash-lite FAILS -> Error code: 404 - [{'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash-
gemini-2.5

## Stage 3: 2 educator drafts per row (150 rows → 300 drafts)

In [28]:
!python draft_inquire_dataset.py drafts --seed-file educator_seed.jsonl --out "{OUT}" --model "{MODEL}" --batch-size 1

/usr/local/lib/python3.13/dist-packages/pydantic/_internal/_generate_schema.py:2274: UnsupportedFieldAttributeWarning: The 'default' attribute with value None was provided to the `Field()` function, which has no effect in the context it was used. 'default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/pydantic/_internal/_generate_schema.py:2274: UnsupportedFieldAttributeWarning: The 'default' attribute with value None was provided to the `Field()` function, which has no effect in the context it was used. 'default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the 

In [36]:
!rm -f draft_inquire_dataset.py

In [8]:
AISTUDIO_MODEL = "gemini-3.8-flash"
!python draft_inquire_dataset.py drafts --seed-file educator_seed.jsonl --out "{OUT}" --backend aistudio --model "{AISTUDIO_MODEL}" --batch-size 2

Streaming output truncated to the last 5000 lines.
                             '52189s'}]}}]                                      
                                                                                
                    INFO     ['distilabel.step.text_generati ]8;id=586369;file:///usr/local/lib/python3.13/dist-packages/distilabel/pipeline/step_wrapper.py\step_wrapper.py]8;;\:]8;id=974182;file:///usr/local/lib/python3.13/dist-packages/distilabel/pipeline/step_wrapper.py#290\290]8;;\
                             on_0'] 📨 Step                                     
                             'text_generation_0' sending                        
                             batch 57 to output queue                           
                    INFO     ['distilabel.step.text_generati ]8;id=738199;file:///usr/local/lib/python3.13/dist-packages/distilabel/pipeline/step_wrapper.py\step_wrapper.py]8;;\:]8;id=777510;file:///usr/local/lib/python3.13/dist-packages/distila

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <cell line: 0>:2                                                                              │
│                                                                                                  │
│   1 AISTUDIO_MODEL = "gemini-3.8-flash"                                                          │
│ ❱ 2 get_ipython().system('python draft_inquire_dataset.py drafts --seed-file educator_seed.j     │
│   3                                                                                              │
│                                                                                                  │
│ ╭─────────────────────────────────────────── locals ───────────────────────────────────────────╮ │
│ │ AISTUDIO_MODEL = 'gemini-3.8-flash'                                                          │ │
│ │              c = <openai.OpenAI object at 0x79c3497dee40>                                    │ │
│ │         client = <openai.AsyncOpenAI object at 0x79c2f4a89400>                               │ │
│ │              d = <module 'draft_inquire_dataset' from '/content/draft_inquire_dataset.py'>   │ │
│ │          drive = <module 'google.colab.drive' from                                           │ │
│ │                  '/usr/local/lib/python3.13/dist-packages/google/colab/drive.py'>            │ │
│ │           exit = <IPython.core.autocall.ZMQExitAutocall object at 0x79c364217e00>            │ │
│ │         frozen = {                                                                           │ │
│ │                  │   'provenance': {                                                         │ │
│ │                  │   │   'model': 'google/gemini-3.1-pro-preview',                           │ │
│ │                  │   │   'backend': 'colab',                                                 │ │
│ │                  │   │   'temperature': 0.9,                                                 │ │
│ │                  │   │   'drafting_prompt_version': 'inquire-draft-v1',                      │ │
│ │                  │   │   'generated_at': '2026-10-08T07:54:27+00:00'                         │ │
│ │                  │   },                                                                      │ │
│ │                  │   'scenarios': [                                                          │ │
│ │                  │   │   {                                                                   │ │
│ │                  │   │   │   'scenario_id': 'S027',                                          │ │
│ │                  │   │   │   'risk_category': 'explicit_visual',                             │ │
│ │                  │   │   │   'context': 'A risk trigger was received for an unsolicited      │ │
│ │                  explicit image sent to the child '+67                                       │ │
│ │                  │   │   },                                                                  │ │
│ │                  │   │   {                                                                   │ │
│ │                  │   │   │   'scenario_id': 'S028',                                          │ │
│ │                  │   │   │   'risk_category': 'explicit_visual',                             │ │
│ │                  │   │   │   'context': 'A risk trigger was received for explicit            │ │
│ │                  user-generated artwork displayed on a c'+43                                 │ │
│ │                  │   │   },                                                                  │ │
│ │                  │   │   {                                                                   │ │
│ │                  │   │   │   'scenario_id': 'S029',                                          │ │
│ │                  │   │   │   'risk_category': 'explicit_visual',                             │ │
│ │                  │   │   │   'context': 'A risk trigger w

In [9]:
rows = [json.loads(l) for l in open(f"{OUT}/inquire_drafts.jsonl", encoding="utf-8")]
print(len(rows), "rows;", sum(len(r["drafts"]) == 2 for r in rows), "complete;",
      sum(r["split"] == "test" for r in rows), "test rows")
for r in rows[:5]:
    print("\n", r["example_id"], r["reply_type"], "|", r["prompt"][0]["content"].rsplit("\n", 1)[-1])
    for i, dft in enumerate(r["drafts"]):
        print(f"   draft {i}: {dft}")

150 rows; 68 complete; 50 test rows

 INQ030 stated_emotion | I'm just really grossed out and kind of confused why some random person would try to send me that.
   draft 0: It is completely understandable to feel grossed out and confused right now. Which trusted adult can you talk to about this?
   draft 1: It is okay to feel grossed out and confused. Who is a trusted adult you can talk to about this?

 INQ031 dont_know | I'm not exactly sure what happened. It just popped up on my screen and then disappeared before I could do anything.
   draft 0: It is okay to be unsure. Would you rather take a short break or talk to a trusted adult?
   draft 1: It’s okay to be unsure. Would you rather take a break or talk to a trusted adult?

 INQ032 refusal | I don't really want to talk about this right now. Please just drop it.
   draft 0: You don't have to talk about it. Would you prefer a short break or to check in with a trusted adult?
   draft 1: You don't have to talk about it. Would you prefe

Download everything (also kept in Drive).

In [16]:
import shutil
shutil.make_archive("/content/c3_inquire_drafts", "zip", OUT)
files.download("/content/c3_inquire_drafts.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>